# NER gia phả — quét cấu hình (không dùng Drive: file đã tải thẳng lên /content)
**Tốn tiền (GPU Colab, ~3 giờ).** Cần có sẵn trong /content: `train_ner_v2.py`, `vi_cv5.zip`. Kết quả ghi ở /content/cv_sweep rồi nén tải về.

In [ ]:
BASE = "/content"

In [ ]:
!pip -q install "transformers>=4.40" sentencepiece
import torch; assert torch.cuda.is_available(), "Chưa bật GPU"
print(torch.cuda.get_device_name(0))
!cd /content && unzip -qo vi_cv5.zip && ls vi_cv5

In [ ]:
import subprocess, json, shutil, os
def sweep(name, model, epochs, lr, batch, seed=42):
    out_dir = f"{BASE}/cv_sweep/{name}"; os.makedirs(out_dir, exist_ok=True)
    for k in range(5):
        if os.path.exists(f"{out_dir}/fold{k}_probs.json"):
            print(f"== {name} fold{k}: đã có, bỏ qua"); continue
        out = f"/content/out_{name}/fold{k}"
        subprocess.run(["python", f"{BASE}/train_ner_v2.py", "--model", model, "--data-dir", f"/content/vi_cv5/fold{k}", "--output-dir", out,
                        "--test-split", "test", "--epochs", str(epochs), "--lr", str(lr), "--batch-size", str(batch), "--seed", str(seed), "--save-probs"], check=True)
        shutil.copy(f"{out}/model_card.json", f"{out_dir}/fold{k}_model_card.json")
        shutil.copy(f"{out}/probs.json", f"{out_dir}/fold{k}_probs.json")   # ghi cuối cùng = dấu hiệu đã xong
        shutil.rmtree(out)
        c = json.load(open(f"{out_dir}/fold{k}_model_card.json"))
        print(f"== {name} fold{k}: best_epoch={c['best_epoch']} dev_F1={c['dev']['micro']['f1']} test_F1={c['test']['micro']['f1']}", flush=True)

In [ ]:
# A) PhoBERT-base chạy lại (kiểm tra tái lập + lưu xác suất)
sweep("phobert_base", "vinai/phobert-base", epochs=20, lr=5e-5, batch=16)

In [ ]:
# B) XLM-RoBERTa-base (MIT)
sweep("xlmr_base", "xlm-roberta-base", epochs=20, lr=3e-5, batch=16)

In [ ]:
# C) PhoBERT-large (MIT) — nặng hơn ~3 lần
sweep("phobert_large", "vinai/phobert-large", epochs=15, lr=2e-5, batch=8)

In [ ]:
!cd /content && zip -qr cv_sweep.zip cv_sweep && ls -la cv_sweep.zip
from google.colab import files
files.download("/content/cv_sweep.zip")
from google.colab import runtime
# runtime.unassign()  # chạy thủ công SAU KHI tải xong